<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW4_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 安裝套件

第一次執行時請取消下面 cell 的註解並安裝需要的套件。若已安裝，可直接跳過。

In [34]:
!pip install pandas requests beautifulsoup4 faiss-cpu sentence-transformers gspread google-auth

## 2. 匯入套件與基本設定

In [35]:
import os
import time
import datetime
import requests
import pandas as pd
import numpy as np

from typing import List, Dict, Any, Optional

# Google Sheet 設定
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing"
GOOGLE_WORKSHEET_NAME = "vul_db"

# API URL
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_API_URL = "https://api.first.org/data/v1/epss"
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"


REQUIRED_COLUMNS = [
    "cve_id",
    "description",
    "published",
    "last_modified",
    "cvss_score",
    "cvss_severity",
    "cwe",
    "vendor",
    "product",
    "epss",
    "kev",
    "known_ransomware",
    "references",
    "risk_priority",
]

## 3. Google Sheets 工具函式

In [36]:
def get_google_worksheet(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    spreadsheet = gc.open_by_url(spreadsheet_url)

    try:
        worksheet = spreadsheet.worksheet(worksheet_name)
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=worksheet_name,
            rows=1000,
            cols=len(REQUIRED_COLUMNS)
        )

    return worksheet


def ensure_google_sheet_schema(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)

    values = worksheet.get_all_values()

    if not values:
        worksheet.append_row(REQUIRED_COLUMNS)
        return "created header"

    header = values[0]

    if header != REQUIRED_COLUMNS:
        worksheet.clear()
        worksheet.append_row(REQUIRED_COLUMNS)
        return "reset header"

    return "ok"


def load_from_google_sheets(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
    fix_schema: bool = True,
) -> pd.DataFrame:
    if fix_schema:
        ensure_google_sheet_schema(spreadsheet_url, worksheet_name)

    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    records = worksheet.get_all_records()

    if not records:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    df = pd.DataFrame(records)

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    return df[REQUIRED_COLUMNS]


def append_to_google_sheets(
    vuln_df: pd.DataFrame,
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    ensure_google_sheet_schema(spreadsheet_url, worksheet_name)
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)

    existing_df = load_from_google_sheets(
        spreadsheet_url,
        worksheet_name,
        fix_schema=False
    )

    existing_ids = set(existing_df["cve_id"].astype(str))

    output_df = vuln_df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in output_df.columns:
            output_df[col] = ""

    output_df = output_df[REQUIRED_COLUMNS]
    output_df = output_df[~output_df["cve_id"].astype(str).isin(existing_ids)]

    if output_df.empty:
        print("沒有新的 CVE 需要寫入。")
        return

    output_df = output_df.fillna("").astype(str)

    worksheet.append_rows(
        output_df.values.tolist(),
        value_input_option="USER_ENTERED"
    )

    print(f"新增 {len(output_df)} 筆資料到 Google Sheets。")

## 4. NVD / EPSS / KEV API 函式

In [37]:
def get_nvd_headers() -> Dict[str, str]:
    headers = {}

    api_key = os.getenv("NVD_API_KEY")
    if api_key:
        headers["apiKey"] = api_key

    return headers


def safe_get_json(url: str, params: Optional[dict] = None, headers: Optional[dict] = None, timeout: int = 60):
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=timeout
        )

        if response.status_code == 429:
            print("遇到 rate limit，等待 30 秒後重試。")
            time.sleep(30)
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

        response.raise_for_status()
        return response.json()

    except Exception as e:
        print(f"API request failed: {e}")
        return None


def fetch_nvd_cve(cve_id: str) -> Optional[dict]:
    params = {
        "cveId": cve_id
    }

    data = safe_get_json(
        NVD_API_URL,
        params=params,
        headers=get_nvd_headers()
    )

    if not data:
        return None

    vulns = data.get("vulnerabilities", [])

    if not vulns:
        return None

    return vulns[0]


def fetch_recent_nvd_cves(days: int = 30, max_pages: int = 3) -> List[dict]:
    end_date = datetime.datetime.utcnow()
    start_date = end_date - datetime.timedelta(days=days)

    params = {
        "lastModStartDate": start_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "lastModEndDate": end_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
        "resultsPerPage": 1000,
        "startIndex": 0,
    }

    headers = get_nvd_headers()
    results = []

    for page in range(max_pages):
        print(f"正在抓 NVD 第 {page + 1} 頁，startIndex={params['startIndex']}")

        data = safe_get_json(
            NVD_API_URL,
            params=params,
            headers=headers
        )

        if not data:
            break

        vulns = data.get("vulnerabilities", [])
        total = data.get("totalResults", 0)

        print(f"本頁取得 {len(vulns)} 筆，NVD totalResults={total}")

        results.extend(vulns)

        params["startIndex"] += params["resultsPerPage"]

        if params["startIndex"] >= total:
            break

        if os.getenv("NVD_API_KEY"):
            time.sleep(1)
        else:
            time.sleep(6)

    return results


def fetch_epss_batch(cve_ids: List[str]) -> Dict[str, dict]:
    if not cve_ids:
        return {}

    result = {}

    batch_size = 100

    for i in range(0, len(cve_ids), batch_size):
        batch = cve_ids[i:i + batch_size]

        params = {
            "cve": ",".join(batch)
        }

        data = safe_get_json(EPSS_API_URL, params=params)

        if not data:
            continue

        for item in data.get("data", []):
            cve = item.get("cve")
            result[cve] = {
                "epss": item.get("epss", ""),
                "percentile": item.get("percentile", "")
            }

        time.sleep(1)

    return result


def fetch_kev_catalog() -> Dict[str, dict]:
    data = safe_get_json(CISA_KEV_URL)

    if not data:
        return {}

    kev_map = {}

    for item in data.get("vulnerabilities", []):
        cve_id = item.get("cveID")

        if cve_id:
            kev_map[cve_id] = item

    return kev_map

## 5. NVD 資料解析函式

In [46]:
def get_english_description(cve_obj: dict) -> str:
    descriptions = cve_obj.get("descriptions", [])

    for desc in descriptions:
        if desc.get("lang") == "en":
            return desc.get("value", "")

    if descriptions:
        return descriptions[0].get("value", "")

    return ""


def extract_cvss(cve_obj: dict):
    metrics = cve_obj.get("metrics", {})

    metric_keys = [
        "cvssMetricV40",
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV2",
    ]

    for key in metric_keys:
        if key in metrics and metrics[key]:
            metric = metrics[key][0]
            cvss_data = metric.get("cvssData", {})

            score = cvss_data.get("baseScore", "")
            severity = cvss_data.get("baseSeverity", "")

            if not severity:
                severity = metric.get("baseSeverity", "")

            return score, severity

    return "", ""


def extract_cwe(cve_obj: dict) -> str:
    weaknesses = cve_obj.get("weaknesses", [])

    cwe_list = []

    for weakness in weaknesses:
        for desc in weakness.get("description", []):
            value = desc.get("value", "")
            if value and value not in cwe_list:
                cwe_list.append(value)

    return ", ".join(cwe_list)


def extract_vendor_product(cve_obj: dict):
    configurations = cve_obj.get("configurations", [])

    vendors = set()
    products = set()

    for config in configurations:
        for node in config.get("nodes", []):
            for cpe_match in node.get("cpeMatch", []):
                criteria = cpe_match.get("criteria", "")

                parts = criteria.split(":")

                if len(parts) >= 5:
                    vendor = parts[3]
                    product = parts[4]

                    if vendor and vendor != "*":
                        vendors.add(vendor)

                    if product and product != "*":
                        products.add(product)

    vendor_text = ", ".join(sorted(vendors))
    product_text = ", ".join(sorted(products))

    return vendor_text, product_text


def extract_references(cve_obj: dict) -> str:
    references = cve_obj.get("references", [])

    # NVD API v2: references 通常是 list
    if isinstance(references, list):
        refs = references

    # 舊格式：references 可能是 dict，裡面有 referenceData
    elif isinstance(references, dict):
        refs = references.get("referenceData", [])

    else:
        refs = []

    urls = []

    for ref in refs[:5]:
        if isinstance(ref, dict):
            url = ref.get("url", "")
            if url:
                urls.append(url)

    return "\n".join(urls)


def calculate_risk_priority(cvss_score, epss, kev: bool) -> str:
    try:
        cvss_score = float(cvss_score)
    except:
        cvss_score = 0.0

    try:
        epss = float(epss)
    except:
        epss = 0.0

    if kev:
        return "CRITICAL"

    if cvss_score >= 9.0 and epss >= 0.1:
        return "CRITICAL"

    if cvss_score >= 9.0:
        return "HIGH"

    if epss >= 0.5:
        return "HIGH"

    if cvss_score >= 7.0 or epss >= 0.1:
        return "MEDIUM"

    return "LOW"


def parse_nvd_vulnerability(vuln: dict, epss_map: dict, kev_map: dict) -> dict:
    cve_obj = vuln.get("cve", {})

    cve_id = cve_obj.get("id", "")
    description = get_english_description(cve_obj)

    published = cve_obj.get("published", "")
    last_modified = cve_obj.get("lastModified", "")

    cvss_score, cvss_severity = extract_cvss(cve_obj)
    cwe = extract_cwe(cve_obj)
    vendor, product = extract_vendor_product(cve_obj)
    references = extract_references(cve_obj)

    epss = epss_map.get(cve_id, {}).get("epss", "")

    kev_item = kev_map.get(cve_id)
    kev = kev_item is not None

    known_ransomware = ""
    if kev_item:
        known_ransomware = kev_item.get("knownRansomwareCampaignUse", "")

    risk_priority = calculate_risk_priority(cvss_score, epss, kev)

    return {
        "cve_id": cve_id,
        "description": description,
        "published": published,
        "last_modified": last_modified,
        "cvss_score": cvss_score,
        "cvss_severity": cvss_severity,
        "cwe": cwe,
        "vendor": vendor,
        "product": product,
        "epss": epss,
        "kev": "YES" if kev else "NO",
        "known_ransomware": known_ransomware,
        "references": references,
        "risk_priority": risk_priority,
    }


def normalize_vulnerability_table(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    output_df = df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in output_df.columns:
            output_df[col] = ""

    output_df = output_df[REQUIRED_COLUMNS]

    output_df["cvss_score"] = pd.to_numeric(output_df["cvss_score"], errors="coerce").fillna(0)
    output_df["epss"] = pd.to_numeric(output_df["epss"], errors="coerce").fillna(0)

    output_df["cve_id"] = output_df["cve_id"].astype(str)

    output_df = output_df.drop_duplicates(subset=["cve_id"], keep="last")

    return output_df

## 6. 爬蟲主流程

In [39]:
def build_dataset_from_nvd_vulns(vulns: List[dict]) -> pd.DataFrame:
    if not vulns:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    cve_ids = []

    for vuln in vulns:
        cve_id = vuln.get("cve", {}).get("id", "")
        if cve_id:
            cve_ids.append(cve_id)

    print(f"準備補 EPSS，共 {len(cve_ids)} 筆 CVE。")
    epss_map = fetch_epss_batch(cve_ids)

    print("準備載入 CISA KEV Catalog。")
    kev_map = fetch_kev_catalog()

    rows = []

    for vuln in vulns:
        row = parse_nvd_vulnerability(vuln, epss_map, kev_map)
        rows.append(row)

    df = pd.DataFrame(rows)
    df = normalize_vulnerability_table(df)

    return df


def build_dataset_from_cves(cve_list: List[str]) -> pd.DataFrame:
    vulns = []

    for i, cve_id in enumerate(cve_list):
        print(f"[{i + 1}/{len(cve_list)}] fetching {cve_id}")

        vuln = fetch_nvd_cve(cve_id)

        if vuln:
            vulns.append(vuln)
        else:
            print(f"找不到資料：{cve_id}")

        if os.getenv("NVD_API_KEY"):
            time.sleep(1)
        else:
            time.sleep(6)

    return build_dataset_from_nvd_vulns(vulns)


def fetch_and_sync_to_sheets(cve_list: List[str]) -> pd.DataFrame:
    print(f"正在從 API 抓取 {len(cve_list)} 筆漏洞資料...")

    new_data_df = build_dataset_from_cves(cve_list)

    if new_data_df.empty:
        print("API 沒有抓到任何資料。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    print("正在將資料附加至 Google Sheets...")
    append_to_google_sheets(new_data_df)

    print("同步完成。")
    display(new_data_df)

    return new_data_df


def crawl_recent_vulnerabilities(days: int = 30, max_pages: int = 1, max_sync: Optional[int] = 20) -> pd.DataFrame:
    print(f"開始抓取最近 {days} 天有更新的 CVE。")

    vulns = fetch_recent_nvd_cves(
        days=days,
        max_pages=max_pages
    )

    if not vulns:
        print("沒有從 NVD 抓到任何漏洞。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    if max_sync is not None:
        vulns = vulns[:max_sync]

    print(f"準備整理 {len(vulns)} 筆漏洞資料。")

    df = build_dataset_from_nvd_vulns(vulns)

    if df.empty:
        print("整理後沒有任何資料。")
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    append_to_google_sheets(df)

    print("最近漏洞同步完成。")
    display(df)

    return df

## 7. RAG 文件轉換

In [40]:
def vulnerability_to_document(row: pd.Series) -> str:
    return f"""
CVE ID: {row.get("cve_id", "")}

Description:
{row.get("description", "")}

Vendor:
{row.get("vendor", "")}

Product:
{row.get("product", "")}

CVSS Score:
{row.get("cvss_score", "")}

CVSS Severity:
{row.get("cvss_severity", "")}

EPSS:
{row.get("epss", "")}

CISA KEV:
{row.get("kev", "")}

Known Ransomware:
{row.get("known_ransomware", "")}

CWE:
{row.get("cwe", "")}

Risk Priority:
{row.get("risk_priority", "")}

References:
{row.get("references", "")}
""".strip()

## 8. 用 FAISS 建立向量搜尋器

In [41]:
import faiss
from sentence_transformers import SentenceTransformer


class FaissRetriever:
    def __init__(self, model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.model = SentenceTransformer(model_name)

        self.index = None
        self.documents = []
        self.ids = []
        self.metadatas = []
        self.dimension = None

    def build(self, documents: list[str], ids: list[str], metadatas: list[dict]):
        self.documents = documents
        self.ids = ids
        self.metadatas = metadatas

        if not documents:
            self.index = None
            self.dimension = None
            return

        embeddings = self.model.encode(
            documents,
            convert_to_numpy=True,
            show_progress_bar=True
        ).astype("float32")

        # normalize 後用 Inner Product，等價於 cosine similarity
        faiss.normalize_L2(embeddings)

        self.dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings)

        print(f"FAISS index 建立完成，共 {self.index.ntotal} 筆資料，dimension={self.dimension}")

    def search(self, query: str, top_k: int = 5):
        if self.index is None or self.index.ntotal == 0:
            return []

        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        scores, indices = self.index.search(query_embedding, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            results.append({
                "score": float(score),
                "id": self.ids[idx],
                "metadata": self.metadatas[idx],
                "document": self.documents[idx],
            })

        return results

## 9. 建立與更新 RAG Index

In [42]:
df = pd.DataFrame(columns=REQUIRED_COLUMNS)
documents = []
ids = []
metadatas = []
retriever = FaissRetriever()


def refresh_rag_index():
    global df, documents, ids, metadatas, retriever

    df = load_from_google_sheets()
    df = normalize_vulnerability_table(df)

    if df.empty:
        print("Google Sheets 裡目前沒有資料，無法建立 RAG index。")
        return

    documents = [vulnerability_to_document(row) for _, row in df.iterrows()]
    ids = df["cve_id"].tolist()

    metadatas = df[
        [
            "cve_id",
            "product",
            "cvss_severity",
            "kev",
            "cwe",
            "risk_priority",
        ]
    ].to_dict("records")

    retriever.build(documents, ids, metadatas)

    print(f"RAG 系統已更新，目前共有 {len(df)} 筆資料。")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [43]:
def search_vulnerabilities(query: str, top_k: int = 5):
    results = retriever.search(query, top_k=top_k)

    if not results:
        return "目前沒有可搜尋的資料，請先執行 crawl_recent_vulnerabilities() 或 fetch_and_sync_to_sheets()。"

    output = []

    for i, item in enumerate(results, start=1):
        meta = item["metadata"]

        text = f"""
## Result {i}

Score: {item["score"]:.4f}

CVE: {meta.get("cve_id", "")}
Product: {meta.get("product", "")}
Severity: {meta.get("cvss_severity", "")}
KEV: {meta.get("kev", "")}
CWE: {meta.get("cwe", "")}
Risk Priority: {meta.get("risk_priority", "")}

{item["document"]}
""".strip()

        output.append(text)

    return "\n\n---\n\n".join(output)

## 10. 第一次測試：確認 Google Sheet 可用

In [44]:
schema_result = ensure_google_sheet_schema()
print("Schema:", schema_result)

loaded_df = load_from_google_sheets()
print("目前 Google Sheets 資料筆數:", len(loaded_df))
display(loaded_df.head())

Schema: reset header
目前 Google Sheets 資料筆數: 0


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority


## 11. 手動抓幾筆確定存在的 CVE

In [47]:
test_df = fetch_and_sync_to_sheets([
    "CVE-2024-42008",
    "CVE-2023-38831",
    "CVE-2024-21413",
    "CVE-2021-41773",
    "CVE-2023-34362",
])

正在從 API 抓取 5 筆漏洞資料...
[1/5] fetching CVE-2024-42008
[2/5] fetching CVE-2023-38831
[3/5] fetching CVE-2024-21413
[4/5] fetching CVE-2021-41773
[5/5] fetching CVE-2023-34362
準備補 EPSS，共 5 筆 CVE。
準備載入 CISA KEV Catalog。
正在將資料附加至 Google Sheets...
新增 5 筆資料到 Google Sheets。
同步完成。


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority
0,CVE-2024-42008,A Cross-Site Scripting vulnerability in rcmail_action_mail_get->run() in Roundcube through 1.5.7 and 1.6.x through 1...,2024-08-05T19:15:38.153,2025-03-13T16:15:21.240,9.3,CRITICAL,CWE-79,roundcube,webmail,0.50951,NO,,https://github.com/roundcube/roundcubemail/releases\nhttps://github.com/roundcube/roundcubemail/releases/tag/1.5.8\n...,CRITICAL
1,CVE-2023-38831,RARLAB WinRAR before 6.23 allows attackers to execute arbitrary code when a user attempts to view a benign file with...,2023-08-23T17:15:43.863,2025-10-31T14:39:33.660,7.8,HIGH,"CWE-345, CWE-351",rarlab,winrar,0.93664,YES,Known,http://packetstormsecurity.com/files/174573/WinRAR-Remote-Code-Execution.html\nhttps://blog.google/threat-analysis-g...,CRITICAL
2,CVE-2024-21413,Microsoft Outlook Remote Code Execution Vulnerability,2024-02-13T18:16:00.137,2025-10-28T14:36:10.643,9.8,CRITICAL,"CWE-20, NVD-CWE-noinfo",microsoft,"365_apps, office_2016, office_2019, office_long_term_servicing_channel",0.92992,YES,Unknown,https://msrc.microsoft.com/update-guide/vulnerability/CVE-2024-21413\nhttps://msrc.microsoft.com/update-guide/vulner...,CRITICAL
3,CVE-2021-41773,A flaw was found in a change made to path normalization in Apache HTTP Server 2.4.49. An attacker could use a path t...,2021-10-05T09:15:07.593,2026-02-17T19:49:26.367,9.8,CRITICAL,CWE-22,"apache, fedoraproject, netapp, oracle","cloud_backup, fedora, http_server, instantis_enterprisetrack",0.94391,YES,Known,http://packetstormsecurity.com/files/164418/Apache-HTTP-Server-2.4.49-Path-Traversal-Remote-Code-Execution.html\nhtt...,CRITICAL
4,CVE-2023-34362,"In Progress MOVEit Transfer before 2021.0.6 (13.0.6), 2021.1.4 (13.1.4), 2022.0.4 (14.0.4), 2022.1.5 (14.1.5), and 2...",2023-06-02T14:15:09.487,2025-10-27T14:37:08.460,9.8,CRITICAL,CWE-89,progress,"moveit_cloud, moveit_transfer",0.94254,YES,Known,http://packetstormsecurity.com/files/172883/MOVEit-Transfer-SQL-Injection-Remote-Code-Execution.html\nhttp://packets...,CRITICAL


## 12. 建立 RAG index

In [48]:
refresh_rag_index()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index 建立完成，共 5 筆資料，dimension=384
RAG 系統已更新，目前共有 5 筆資料。


## 13. 測試搜尋

In [49]:
print(search_vulnerabilities("WinRAR 有哪些高風險漏洞？", top_k=3))

## Result 1

Score: 0.4606

CVE: CVE-2024-21413
Product: 365_apps, office_2016, office_2019, office_long_term_servicing_channel
Severity: CRITICAL
KEV: YES
CWE: CWE-20, NVD-CWE-noinfo
Risk Priority: CRITICAL

CVE ID: CVE-2024-21413

Description:
Microsoft Outlook Remote Code Execution Vulnerability

Vendor:
microsoft

Product:
365_apps, office_2016, office_2019, office_long_term_servicing_channel

CVSS Score:
9.8

CVSS Severity:
CRITICAL

EPSS:
0.92992

CISA KEV:
YES

Known Ransomware:
Unknown

CWE:
CWE-20, NVD-CWE-noinfo

Risk Priority:
CRITICAL

References:
https://msrc.microsoft.com/update-guide/vulnerability/CVE-2024-21413
https://msrc.microsoft.com/update-guide/vulnerability/CVE-2024-21413
https://research.checkpoint.com/2024/the-risks-of-the-monikerlink-bug-in-microsoft-outlook-and-the-big-picture/
https://www.vicarius.io/vsociety/posts/cve-2024-21413-critical-monikerlink-vulnerability-affecting-microsoft-outlook-detection-script
https://www.vicarius.io/vsociety/posts/cve-2024-21

## 14. 真的開始爬最近漏洞


In [50]:
recent_df = crawl_recent_vulnerabilities(
    days=30,
    max_pages=1,
    max_sync=10
)

開始抓取最近 30 天有更新的 CVE。
正在抓 NVD 第 1 頁，startIndex=0


/tmp/ipykernel_1514/3765722997.py:61: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.datetime.utcnow()


本頁取得 1000 筆，NVD totalResults=43769
準備整理 10 筆漏洞資料。
準備補 EPSS，共 10 筆 CVE。
準備載入 CISA KEV Catalog。
新增 10 筆資料到 Google Sheets。
最近漏洞同步完成。


,cve_id,description,published,last_modified,cvss_score,cvss_severity,cwe,vendor,product,epss,kev,known_ransomware,references,risk_priority
0,CVE-1999-0511,IP forwarding is enabled on a machine which is not a router or firewall.,1997-01-01T05:00:00.000,2026-05-28T18:16:20.797,9.1,CRITICAL,"NVD-CWE-Other, CWE-200",microsoft,"windows_2000, windows_nt",0.04827,NO,,https://www.cve.org/CVERecord?id=CVE-1999-0511\nhttps://www.cve.org/CVERecord?id=CVE-1999-0511,HIGH
1,CVE-1999-0517,"An SNMP community name is the default (e.g. public), null, or missing.",1997-01-01T05:00:00.000,2026-05-28T18:16:21.023,5.9,MEDIUM,"NVD-CWE-Other, CWE-200","hp, sun","hp-ux, sunos",0.89585,NO,,https://exchange.xforce.ibmcloud.com/vulnerabilities/CVE-1999-0517\nhttps://exchange.xforce.ibmcloud.com/vulnerabili...,HIGH
2,CVE-1999-0524,ICMP information such as (1) netmask and (2) timestamp is allowed from arbitrary hosts.,1997-08-01T04:00:00.000,2026-05-28T18:16:21.263,4.0,MEDIUM,"CWE-200, NVD-CWE-noinfo","apple, cisco, hp, ibm, linux, microsoft, novell, oracle, sco, sgi, windriver","aix, bsdos, hp-ux, ios, irix, linux_kernel, mac_os_x, macos, netware, os2, sco_unix, solaris, tru64, windows",0.00299,NO,,http://descriptions.securescout.com/tc/11010\nhttp://descriptions.securescout.com/tc/11011\nhttp://kb.juniper.net/In...,LOW
3,CVE-1999-0632,The RPC portmapper service is running.,1999-01-01T05:00:00.000,2026-05-28T18:16:21.613,7.3,HIGH,"NVD-CWE-Other, CWE-200",,,0.00875,NO,,https://www.cve.org/CVERecord?id=CVE-1999-0632\nhttps://www.cve.org/CVERecord?id=CVE-1999-0632,MEDIUM
4,CVE-2004-2320,"The default configuration of BEA WebLogic Server and Express 8.1 SP2 and earlier, 7.0 SP4 and earlier, 6.1 through S...",2004-12-31T05:00:00.000,2026-05-28T19:16:21.473,5.3,MEDIUM,CWE-200,bea,weblogic_server,0.04148,NO,,http://dev2dev.bea.com/pub/advisory/68\nhttp://secunia.com/advisories/10726\nhttp://www.kb.cert.org/vuls/id/867593\n...,LOW
5,CVE-2005-1794,Microsoft Terminal Server using Remote Desktop Protocol (RDP) 5.2 stores an RSA private key in mstlsapi.dll and uses...,2005-06-01T04:00:00.000,2026-05-22T11:16:19.077,7.4,HIGH,NVD-CWE-Other,microsoft,"remote_desktop_connection, windows_terminal_services_using_rdp",0.05970,NO,,http://secunia.com/advisories/15605/\nhttp://www.oxid.it/downloads/rdp-gbu.pdf\nhttp://www.securityfocus.com/bid/138...,MEDIUM
6,CVE-2008-4250,"The Server service in Microsoft Windows 2000 SP4, XP SP2 and SP3, Server 2003 SP1 and SP2, Vista Gold and SP1, Serve...",2008-10-23T22:00:01.357,2026-05-21T12:57:17.353,9.8,CRITICAL,"CWE-94, CWE-119",microsoft,"windows_2000, windows_server_2003, windows_server_2008, windows_vista, windows_xp",0.92078,YES,Unknown,http://blogs.securiteam.com/index.php/archives/1150\nhttp://marc.info/?l=bugtraq&m=122703006921213&w=2\nhttp://secun...,CRITICAL
7,CVE-2008-4309,"Integer overflow in the netsnmp_create_subtree_cache function in agent/snmp_agent.c in net-snmp 5.4 before 5.4.2.1, ...",2008-10-31T20:29:09.497,2026-05-28T19:16:22.167,7.5,HIGH,"CWE-20, CWE-190",net-snmp,net-snmp,0.11399,NO,,http://lists.apple.com/archives/security-announce/2009/May/msg00002.html\nhttp://lists.apple.com/archives/security-a...,MEDIUM
8,CVE-2008-5161,"Error handling in the SSH protocol in (1) SSH Tectia Client and Server and Connector 4.0 through 4.4.11, 5.0 through...",2008-11-19T17:30:00.670,2026-05-28T19:16:22.650,3.7,LOW,"CWE-200, CWE-329","openbsd, ssh","openssh, tectia_client, tectia_connector, tectia_connectsecure, tectia_server",0.01602,NO,,http://isc.sans.org/diary.html?storyid=5366\nhttp://kb.juniper.net/InfoCenter/index?page=content&id=JSA10705\nhttp:/...,LOW
9,CVE-2004-2761,"The MD5 Message-Digest Algorithm is not collision resistant, which makes it easier for context-dependent attackers t...",2009-01-05T20:30:02.140,2026-05-28T19:16:21.770,9.8,CRITICAL,"CWE-310, CWE-328",ietf,"md5, x.509_certificate",0.08251,NO,,http://blog.mozilla.com/security/2008/12/30/md5-weaknesses-could-lead-to-certificate-forgery/\nhttp://blogs.technet...

In [51]:
refresh_rag_index()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index 建立完成，共 15 筆資料，dimension=384
RAG 系統已更新，目前共有 15 筆資料。


In [52]:
print(search_vulnerabilities("最近有哪些最值得優先修補的漏洞？", top_k=5))

## Result 1

Score: 0.3662

CVE: CVE-2004-2761
Product: md5, x.509_certificate
Severity: CRITICAL
KEV: NO
CWE: CWE-310, CWE-328
Risk Priority: HIGH

CVE ID: CVE-2004-2761

Description:
The MD5 Message-Digest Algorithm is not collision resistant, which makes it easier for context-dependent attackers to conduct spoofing attacks, as demonstrated by attacks on the use of MD5 in the signature algorithm of an X.509 certificate.

Vendor:
ietf

Product:
md5, x.509_certificate

CVSS Score:
9.8

CVSS Severity:
CRITICAL

EPSS:
0.08251

CISA KEV:
NO

Known Ransomware:


CWE:
CWE-310, CWE-328

Risk Priority:
HIGH

References:
http://blog.mozilla.com/security/2008/12/30/md5-weaknesses-could-lead-to-certificate-forgery/
http://blogs.technet.com/swi/archive/2008/12/30/information-regarding-md5-collisions-problem.aspx
http://secunia.com/advisories/33826
http://secunia.com/advisories/34281
http://secunia.com/advisories/42181

---

## Result 2

Score: 0.3302

CVE: CVE-2024-21413
Product: 365_apps, office